# COSC 2671 — Assignment 2
## Notebook 1: Data Collection
**Project:** Who Drives Open-Source AI? Network Centrality and Discourse Analysis on GitHub ML Repositories
**Source:** GitHub REST API via PyGitHub
**Repos:** scikit-learn, huggingface/transformers, pytorch, keras, langchain



**Author:** 
Rob De Guzman
Phavan
Group [XX] — RMIT University, Semester 1 2026

***Cell 1 Imports and File Verification***

In [3]:
import subprocess, os
from pathlib import Path

PROJECT_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True).strip())
os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")
print(f"data/raw exists: {os.path.exists('data/raw')}")

Working directory: /Users/rob/Desktop/2026/social/SMNA2026-A2-github-networks
data/raw exists: True


***Cell 2 — Install Dependencies (run once)***

In [3]:
import subprocess, sys

packages = [
    "PyGithub",
    "python-dotenv",
    "pandas",
    "numpy",
    "networkx",
    "python-louvain",
    "bertopic",
    "sentence-transformers",
    "vaderSentiment",
    "pyvis",
    "matplotlib",
    "seaborn",
    "plotly",
    "scipy",
    "tqdm",
]

print("Installing packages...\n")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status}  {pkg}")

print("\nAll packages installed. Restart kernel if this is your first run.")

Installing packages...

  ✅  PyGithub
  ✅  python-dotenv
  ✅  pandas
  ✅  numpy
  ✅  networkx
  ✅  python-louvain
  ✅  bertopic
  ✅  sentence-transformers
  ✅  vaderSentiment
  ✅  pyvis
  ✅  matplotlib
  ✅  seaborn
  ✅  plotly
  ✅  scipy
  ✅  tqdm

All packages installed. Restart kernel if this is your first run.


***Cell 3 — Imports & Auth***

In [12]:
from github import Github, Auth
from dotenv import load_dotenv
import os, time, json
from tqdm import tqdm

# Load token from .env file
load_dotenv()
token = os.getenv("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN not found in .env file.")

# Fix 1: use Auth.Token() — no deprecation warning
g = Github(auth=Auth.Token(token))

# Verify authentication
user = g.get_user()
print(f"Authenticated as: {user.login}")

# Fix 2: rate_limiting property works across all PyGithub versions
remaining, limit = g.rate_limiting
print(f"Rate limit remaining: {remaining} / {limit}")

Authenticated as: s3944714
Rate limit remaining: 4999 / 5000


***Cell 4- Config***

In [11]:
# Repositories to collect from
REPOS = [
    "scikit-learn/scikit-learn",
    "huggingface/transformers",
    "pytorch/pytorch",
    "keras-team/keras",
    "langchain-ai/langchain"
]

# How many issues to collect per repo
# Set lower (e.g. 500) for a test run, 2000 for full collection
MAX_ISSUES = 2000

# Output directory
RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

print(f"Will collect up to {MAX_ISSUES} issues from {len(REPOS)} repos")
print(f"Saving raw JSON to: {RAW_DIR}/")

Will collect up to 2000 issues from 5 repos
Saving raw JSON to: data/raw/


***Cell 5 — Rate Limit Helper***

In [13]:
def check_rate_limit(g, min_remaining=100, sleep_seconds=65):
    """
    Check GitHub API rate limit.
    If below min_remaining, sleep until reset.
    """
    remaining, limit = g.rate_limiting

    if remaining < min_remaining:
        print(f"\n⚠️  Rate limit low ({remaining} remaining). Sleeping {sleep_seconds}s...")
        time.sleep(sleep_seconds)
        print("Resuming collection.")

    return g.rate_limiting[0]

***Cell 6 — Collection Function***

In [14]:
def collect_repo_issues(g, repo_name, max_issues=2000):
    """
    Collect issues and their comments from a GitHub repository.

    Returns a list of dicts, one per issue, each containing:
    - issue metadata (number, title, body, state, author, dates, labels)
    - a list of comments (author, body, created_at)
    """
    print(f"\n{'='*60}")
    print(f"Collecting: {repo_name}")
    print(f"{'='*60}")

    repo = g.get_repo(repo_name)
    print(f"Repo found: {repo.full_name} | Stars: {repo.stargazers_count:,}")

    issues_data = []
    issue_iterator = repo.get_issues(state="all", sort="created", direction="desc")

    collected = 0
    for issue in tqdm(issue_iterator, desc="Issues", total=max_issues):
        if collected >= max_issues:
            break

        # Skip pull requests (GitHub API returns PRs in issues endpoint)
        if issue.pull_request is not None:
            continue

        # Check rate limit every 200 issues
        if collected % 200 == 0 and collected > 0:
            remaining = check_rate_limit(g, min_remaining=100)
            print(f"  [Rate limit: {remaining} remaining]")

        # Collect comments for this issue
        comments_data = []
        try:
            for comment in issue.get_comments():
                comments_data.append({
                    "comment_id": comment.id,
                    "author":     comment.user.login if comment.user else None,
                    "body":       comment.body,
                    "created_at": str(comment.created_at)
                })
                time.sleep(0.1)  # small delay to be gentle on API
        except Exception as e:
            print(f"  Warning: Could not fetch comments for issue #{issue.number}: {e}")

        issues_data.append({
            "issue_number":  issue.number,
            "title":         issue.title,
            "body":          issue.body,
            "state":         issue.state,
            "author":        issue.user.login if issue.user else None,
            "created_at":    str(issue.created_at),
            "closed_at":     str(issue.closed_at) if issue.closed_at else None,
            "labels":        [label.name for label in issue.labels],
            "comment_count": issue.comments,
            "comments":      comments_data
        })

        collected += 1
        time.sleep(0.3)  # respect rate limits

    print(f"\n✅ Collected {len(issues_data)} issues from {repo_name}")
    total_comments = sum(len(i["comments"]) for i in issues_data)
    print(f"   Total comments collected: {total_comments:,}")
    return issues_data

***Cell 7 — Run Collection***

In [15]:
# ⚠️ This cell will take several hours for all 5 repos at MAX_ISSUES=2000
# Run overnight or reduce MAX_ISSUES for a quick test

collection_summary = {}

for repo_name in REPOS:
    safe_name   = repo_name.replace("/", "_")
    output_path = os.path.join(RAW_DIR, f"{safe_name}_issues.json")

    # Skip if already collected
    if os.path.exists(output_path):
        print(f"⏭️  Already collected: {repo_name} — skipping (delete file to re-collect)")
        with open(output_path) as f:
            data = json.load(f)
        collection_summary[repo_name] = {
            "issues":   len(data),
            "comments": sum(len(i["comments"]) for i in data)
        }
        continue

    try:
        data = collect_repo_issues(g, repo_name, max_issues=MAX_ISSUES)

        # Save raw JSON
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        collection_summary[repo_name] = {
            "issues":   len(data),
            "comments": sum(len(i["comments"]) for i in data)
        }
        print(f"💾 Saved to {output_path}")

    except Exception as e:
        print(f"❌ Failed on {repo_name}: {e}")
        collection_summary[repo_name] = {"issues": 0, "comments": 0, "error": str(e)}

print("\n\n" + "="*60)
print("COLLECTION COMPLETE — SUMMARY")
print("="*60)
for repo, stats in collection_summary.items():
    print(f"{repo:40} | Issues: {stats.get('issues', 0):5} | Comments: {stats.get('comments', 0):6}")


Collecting: scikit-learn/scikit-learn
Repo found: scikit-learn/scikit-learn | Stars: 66,073


Issues:  46%|█████████████████████████████████▎                                      | 925/2000 [07:17<04:12,  4.26it/s]

  [Rate limit: 4798 remaining]


Issues:  93%|█████████████████████████████████████████████████████████████████▉     | 1856/2000 [15:47<01:46,  1.35it/s]

  [Rate limit: 4598 remaining]


...

  [Rate limit: 4663 remaining]


Issues: 8684it [1:01:01,  1.32it/s]

  [Rate limit: 4463 remaining]


Issues: 9465it [1:07:47,  2.33it/s]


✅ Collected 2000 issues from langchain-ai/langchain
   Total comments collected: 6,500
💾 Saved to data/raw/langchain-ai_langchain_issues.json


COLLECTION COMPLETE — SUMMARY
scikit-learn/scikit-learn                | Issues:  2000 | Comments:  10199
huggingface/transformers                 | Issues:  2000 | Comments:   8286
pytorch/pytorch                          | Issues:  2000 | Comments:   5051
keras-team/keras                         | Issues:  2000 | Comments:   9371
langchain-ai/langchain                   | Issues:  2000 | Comments:   6500


***Cell 8 — Verify Output***

In [16]:
import json

print("Verifying collected files...\n")

for repo_name in REPOS:
    safe_name = repo_name.replace("/", "_")
    path = f"data/raw/{safe_name}_issues.json"
    
    if not os.path.exists(path):
        print(f"❌ MISSING: {path}")
        continue
    
    with open(path) as f:
        data = json.load(f)
    
    total_comments = sum(len(i["comments"]) for i in data)
    authors = set(i["author"] for i in data if i["author"])
    commenters = set(
        c["author"]
        for i in data
        for c in i["comments"]
        if c["author"]
    )
    
    print(f"✅ {repo_name}")
    print(f"   Issues:          {len(data):,}")
    print(f"   Total comments:  {total_comments:,}")
    print(f"   Issue authors:   {len(authors):,}")
    print(f"   Commenters:      {len(commenters):,}")
    print(f"   File size:       {os.path.getsize(path) / 1e6:.1f} MB")
    
    # Show a sample issue
    sample = data[0]
    print(f"   Sample issue:    #{sample['issue_number']} — {sample['title'][:60]}")
    print()

Verifying collected files...

✅ scikit-learn/scikit-learn
   Issues:          2,000
   Total comments:  10,199
   Issue authors:   971
   Commenters:      1,523
   File size:       13.9 MB
   Sample issue:    #34006 — RFC/META Harden CI security

✅ huggingface/transformers
   Issues:          2,000
   Total comments:  8,286
   Issue authors:   1,470
   Commenters:      1,796
   File size:       12.6 MB
   Sample issue:    #45950 — BAN

✅ pytorch/pytorch
   Issues:          2,000
   Total comments:  5,051
   Issue authors:   627
   Commenters:      673
   File size:       15.2 MB
   Sample issue:    #183708 — torch._logging is incomplete

✅ keras-team/keras
   Issues:          2,000
   Total comments:  9,371
   Issue authors:   1,147
   Commenters:      1,312
   File size:       10.9 MB
   Sample issue:    #22887 — keras.ops.conv_transpose skips input-channel validation in s

✅ langchain-ai/langchain
   Issues:          2,000
   Total comments:  6,500
   Issue authors:   1,566
   Commen